# EDA: Captions & Justifications
Adapts analyzing-persona-effects-mllm/notebooks/01_eda_captions_justifications.ipynb
for the cultural VLM annotation outputs.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.analysis.data_loading import load_annotations, load_failures, parse_failure_rate

sns.set_theme(style='whitegrid', palette='colorblind')

In [ ]:
# Load all annotations
df = load_annotations()
failures = load_failures()
print(f'Annotations: {len(df):,}  |  Failures: {len(failures):,}  |  Failure rate: {parse_failure_rate(df, failures):.2%}')
df.head()

In [ ]:
# Sentiment distribution per culture × model
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, model in zip(axes, df['model_name'].unique()[:3]):
    sub = df[df['model_name'] == model]
    ct = sub.groupby(['culture', 'predicted_sentiment']).size().unstack(fill_value=0)
    ct.plot(kind='bar', ax=ax, colormap='RdYlGn')
    ax.set_title(f'{model} — Sentiment Distribution by Culture')
    ax.set_xlabel('Culture'); ax.set_ylabel('Count')
    ax.legend(title='Sentiment', labels=['neg','sl_neg','neu','sl_pos','pos'])
plt.tight_layout(); plt.savefig('../outputs/figures/01_sentiment_distribution.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Caption and justification length distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
df.groupby('culture')['caption_len'].mean().sort_values().plot(kind='barh', ax=ax1)
ax1.set_title('Mean Caption Length by Culture'); ax1.set_xlabel('Words')
df.groupby('culture')['justification_len'].mean().sort_values().plot(kind='barh', ax=ax2)
ax2.set_title('Mean Justification Length by Culture'); ax2.set_xlabel('Words')
plt.tight_layout(); plt.savefig('../outputs/figures/01_text_lengths.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Perception tag frequency (top 20)
from collections import Counter
all_tags = [t for tags in df['predicted_perceptions'].dropna() for t in (tags if isinstance(tags, list) else [])]
tag_counts = Counter(all_tags).most_common(20)
fig, ax = plt.subplots(figsize=(10, 6))
tags_, counts_ = zip(*tag_counts)
ax.barh(tags_, counts_); ax.set_title('Top 20 Perception Tags'); ax.invert_yaxis()
plt.tight_layout(); plt.savefig('../outputs/figures/01_perception_tags.pdf', bbox_inches='tight')
plt.show()